In [4]:
import numpy as np
import struct
import tensorflow as tf
from tensorflow.keras import layers, models


def load_mnist_images(filepath):
    with open(filepath, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8)
        images = images.reshape(num, rows * cols)
    return images

def load_mnist_labels(filepath):
    with open(filepath, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

X_train = load_mnist_images('train-images.idx3-ubyte')
y_train = load_mnist_labels('train-labels.idx1-ubyte')
X_test  = load_mnist_images('t10k-images.idx3-ubyte')
y_test  = load_mnist_labels('t10k-labels.idx1-ubyte')
y_train_oh = tf.keras.utils.to_categorical(y_train, 10)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  10)


# 2. Veriyi CNN'e uygun hale getirme (Reshape)
# MLP'de (60000, 784) yapıyorduk, CNN için (60000, 28, 28, 1) olmalı

In [2]:
X_train_cnn = X_train.reshape((-1, 28, 28, 1))
X_test_cnn = X_test.reshape((-1, 28, 28, 1))

# 3. CNN Model Mimarisi
cnn_model = models.Sequential([
    # İlk Evrişim Katmanı: 32 adet 3x3 filtre
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)), # Görüntüyü 14x14'e düşürür
    
    # İkinci Evrişim Katmanı: 64 adet 3x3 filtre
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)), # Görüntüyü 7x7'ye düşürür
    
    # Düzleştirme ve MLP Kısmı
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.summary()

c:\Python\Python3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_cnn = cnn_model.fit(
    X_train_cnn, y_train_oh,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

test_loss, test_acc = cnn_model.evaluate(X_test_cnn, y_test_oh, verbose=0)
print(f"\nCNN Test Accuracy: {test_acc:.4f}")
print(f"MLP Test Accuracy: 0.9740")
print(f"Fark: {test_acc - 0.9740:.4f}")

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9261 - loss: 0.4659 - val_accuracy: 0.9838 - val_loss: 0.0640
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.9794 - loss: 0.0702 - val_accuracy: 0.9817 - val_loss: 0.0658
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.9850 - loss: 0.0505 - val_accuracy: 0.9852 - val_loss: 0.0614
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.9886 - loss: 0.0369 - val_accuracy: 0.9848 - val_loss: 0.0660
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.9899 - loss: 0.0334 - val_accuracy: 0.9850 - val_loss: 0.0580
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9909 - loss: 0.0284 - val_accuracy: 0.9870 - val_loss: 0.0622
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9923 - loss: 0.0250 - val_accuracy: 0.9880 - val_loss: 0.0502
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9920 - loss: 0.0242 - val_

In [7]:
# CNN tahminleri
y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn), axis=1)

# Hata karşılaştırması
from collections import defaultdict

errors_cnn = defaultdict(list)
for pred, true in zip(y_pred_cnn, y_test):
    if pred != true:
        errors_cnn[true].append(pred)

print(f"{'Rakam':<8} {'MLP Hata':>10} {'CNN Hata':>10} {'Fark':>8}")
print("-" * 40)
mlp_errors = {0: 13, 1: 13, 2: 30, 3: 26, 4: 21, 5: 31, 6: 27, 7: 33, 8: 30, 9: 36}
for digit in range(10):
    mlp_e = mlp_errors[digit]
    cnn_e = len(errors_cnn.get(digit, []))
    print(f"{digit:<8} {mlp_e:>10} {cnn_e:>10} {mlp_e - cnn_e:>+8}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Rakam      MLP Hata   CNN Hata     Fark
----------------------------------------
0                13          3      +10
1                13          5       +8
2                30         17      +13
3                26          6      +20
4                21          7      +14
5                31          9      +22
6                27         15      +12
7                33         17      +16
8                30         26       +4
9                36         30       +6
